In [109]:
!pip install pypdf requests langchain

In [110]:
import requests
import os

# Create a folder to keep our PDFs tidy
os.makedirs("pdfs", exist_ok=True)

# The 5 papers from your assignment, with nice names
papers = {
    "attention.pdf":  "https://arxiv.org/pdf/1706.03762.pdf",
    "bert.pdf":       "https://arxiv.org/pdf/1810.04805.pdf",
    "gpt3.pdf":       "https://arxiv.org/pdf/2005.14165.pdf",
    "roberta.pdf":    "https://arxiv.org/pdf/1907.11692.pdf",
    "t5.pdf":         "https://arxiv.org/pdf/1910.10683.pdf",
}

# Loop through each paper and download it
for filename, url in papers.items():
    path = os.path.join("pdfs", filename)
    print(f"Downloading {filename} ...")
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    with open(path, "wb") as f:
        f.write(response.content)
    print(f"  Saved to {path} ({len(response.content)} bytes)")

print("\nAll downloads complete!")


  Saved to pdfs/attention.pdf (2215244 bytes)
  Saved to pdfs/bert.pdf (775166 bytes)
  Saved to pdfs/gpt3.pdf (6768044 bytes)
  Saved to pdfs/roberta.pdf (209675 bytes)
  Saved to pdfs/t5.pdf (1190021 bytes)

All downloads complete!


In [111]:
from pypdf import PdfReader

# We'll store each paper's text along with its name (metadata)
documents = []

for filename in papers.keys():
    path = os.path.join("pdfs", filename)
    reader = PdfReader(path)

    # Join the text from every page into one big string
    full_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:                       # some pages may be empty
            full_text += text + "\n"

    documents.append({
        "source": filename,            # remember which paper this is
        "text": full_text
    })
    print(f"{filename}: extracted {len(full_text)} characters")

print("\nText extraction done!")


attention.pdf: extracted 39612 characters
bert.pdf: extracted 64140 characters
gpt3.pdf: extracted 236749 characters
roberta.pdf: extracted 49009 characters
t5.pdf: extracted 209772 characters

Text extraction done!


In [112]:
import re

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)      # collapse multiple spaces/newlines into one space
    text = text.strip()                    # trim leading/trailing spaces
    return text

for doc in documents:
    doc["text"] = clean_text(doc["text"])

print("Cleaned all documents.")
print("\nPreview of the Attention paper:\n")
print(documents[0]["text"][:500])          # show first 500 characters


Cleaned all documents.

Preview of the Attention paper:

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz 


In [113]:
!pip install langchain

In [114]:
!pip install langchain-text-splitters

In [115]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# This tool splits text smartly at sentence/paragraph boundaries
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # each chunk ~500 characters
    chunk_overlap=50,     # chunks overlap by 50 chars so we don't cut ideas in half
)

all_chunks = []   # this will hold every chunk from all 5 papers

for doc in documents:
    pieces = splitter.split_text(doc["text"])
    for piece in pieces:
        all_chunks.append({
            "source": doc["source"],   # keep track of which paper it came from
            "text": piece
        })

print(f"Created {len(all_chunks)} chunks from {len(documents)} papers.")
print("\nExample chunk:\n")
print(all_chunks[0]["text"])


Created 1335 chunks from 5 papers.

Example chunk:

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ Google Research llion@google.com Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu Łukasz


In [116]:
!pip install sentence-transformers faiss-cpu

In [117]:
from sentence_transformers import SentenceTransformer

# A small, fast, free model that's great for beginners
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")
print("Each chunk will become a vector of length:", embed_model.get_sentence_embedding_dimension())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!
Each chunk will become a vector of length: 384


/tmp/ipykernel_1287/1928929372.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Each chunk will become a vector of length:", embed_model.get_sentence_embedding_dimension())


In [118]:
# Pull just the text out of each chunk into a simple list
chunk_texts = [chunk["text"] for chunk in all_chunks]

print(f"Embedding {len(chunk_texts)} chunks... (this may take a minute)")

# Convert every chunk into a vector
embeddings = embed_model.encode(
    chunk_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("\nDone! Shape of our embeddings:", embeddings.shape)


Embedding 1335 chunks... (this may take a minute)


Batches:   0%|          | 0/42 [00:00<?, ?it/s]


Done! Shape of our embeddings: (1335, 384)


In [119]:
import faiss
import numpy as np

# How long each vector is (384 for our model)
dimension = embeddings.shape[1]

# Create a FAISS index that searches by similarity
index = faiss.IndexFlatL2(dimension)

# Add all our chunk vectors into the index
index.add(embeddings)

print("FAISS index built!")
print("Number of vectors stored:", index.ntotal)


FAISS index built!
Number of vectors stored: 1335


In [120]:
def search(query, top_k=3):
    # 1. Turn the question into a vector (same model as the chunks!)
    query_vector = embed_model.encode([query], convert_to_numpy=True)

    # 2. Ask FAISS for the closest chunks
    distances, indices = index.search(query_vector, top_k)

    # 3. Show the matching chunks
    print(f"Question: {query}\n")
    for rank, idx in enumerate(indices[0]):
        chunk = all_chunks[idx]
        print(f"--- Match #{rank+1} (from {chunk['source']}) ---")
        print(chunk["text"][:300], "...\n")

# Try it!
search("What is the self-attention mechanism?")


Question: What is the self-attention mechanism?

--- Match #1 (from attention.pdf) ---
to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating dif ...

--- Match #2 (from attention.pdf) ---
to the combination of a self-attention layer and a point-wise feed-forward layer, the approach we take in our model. As side benefit, self-attention could yield more interpretable models. We inspect attention distributions from our models and present and discuss examples in the appendix. Not only do ...

--- Match #3 (from t5.pdf) ---
x5 y5 y4 y3 y2 y1 Figure 3: Matrices representing different attention mask patterns. The input and output of the self-attention mechanism are denotedx and y respectively. A dark cell at rowi and columnj indicates that the self-atten

In [121]:
import pickle

# Save the FAISS index
faiss.write_index(index, "faiss_index.bin")

# Save the chunks (text + source) so we can look them up later
with open("chunks.pkl", "wb") as f:
    pickle.dump(all_chunks, f)

print("Saved! You now have faiss_index.bin and chunks.pkl")


Saved! You now have faiss_index.bin and chunks.pkl


In [122]:
!pip install transformers torch

In [123]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

# Load the tokenizer (turns text into numbers the model understands)
tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")

# Load the actual model
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

print("Language model loaded and ready!")



Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Language model loaded and ready!


In [124]:
def answer_question(query, top_k=3):
    # --- STEP 1: Retrieve relevant chunks (reusing Phase 2) ---
    query_vector = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, top_k)

    retrieved_chunks = [all_chunks[idx]["text"] for idx in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # --- STEP 2: Build the prompt (context + question) ---
    prompt = f"""Use the following context to answer the question.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {query}

Answer:"""

    # --- STEP 3: Generate the answer with the LLM ---
    # Convert the prompt text into numbers (tokens)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    # Ask the model to generate an answer
    output_ids = model.generate(**inputs, max_new_tokens=200)

    # Convert the numbers back into readable text
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    return answer, retrieved_chunks

print("RAG function ready!")


RAG function ready!


In [125]:
question = "What is the self-attention mechanism in transformers?"

answer, sources = answer_question(question)

print("QUESTION:", question)
print("\nANSWER:", answer)
print("\n--- Based on these retrieved chunks ---")
for i, chunk in enumerate(sources):
    print(f"\nChunk {i+1}: {chunk[:200]}...")


QUESTION: What is the self-attention mechanism in transformers?

ANSWER: The output of the final decoder block is fed into a block of the Transformer is self-attention

--- Based on these retrieved chunks ---

Chunk 1: except that it includes a standard attention 3. http://nlp.seas.harvard.edu/2018/04/03/attention.html 4. http://jalammar.github.io/illustrated-transformer/ 4 Exploring the Limits of Transfer Learning ...

Chunk 2: block of the Transformer is self-attention (Cheng et al., 2016). Self-attention is a variant of attention (Graves, 2013; Bahdanau et al., 2015) that processes a sequence by replacing each element by a...

Chunk 3: output of the final decoder block is fed into a dense layer with a softmax output, whose weights are shared with the input embedding matrix. All attention mechanisms in the Transformer are split up in...


In [126]:
# This list will hold our recent conversation turns
# Each turn is a dictionary: {"question": ..., "answer": ...}
conversation_memory = []

MAX_TURNS = 4   # the assignment wants the last 4 interactions

print("Memory initialized! Currently holding", len(conversation_memory), "turns.")


Memory initialized! Currently holding 0 turns.


In [127]:
def format_memory():
    if not conversation_memory:
        return "No previous conversation yet."

    history = ""
    for turn in conversation_memory:
        history += f"Previous Question: {turn['question']}\n"
        history += f"Previous Answer: {turn['answer']}\n\n"
    return history

print("Memory formatter ready!")


Memory formatter ready!


In [128]:
def chat(query, top_k=3):
    # --- STEP 1: Retrieve relevant chunks from the PDFs ---
    query_vector = embed_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_vector, top_k)
    retrieved_chunks = [all_chunks[idx]["text"] for idx in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # --- STEP 2: Get the recent conversation history ---
    history = format_memory()

    # --- STEP 3: Build a prompt with BOTH history and context ---
    prompt = f"""You are a helpful assistant answering questions about AI research papers.
Use the conversation history and the context below to answer the question.
If the answer is not in the context, say you don't know.

Conversation History:
{history}

Context from documents:
{context}

Current Question: {query}

Answer:"""

    # --- STEP 4: Generate the answer ---
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    output_ids = model.generate(**inputs, max_new_tokens=200)
    answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # --- STEP 5: Save this turn into memory ---
    conversation_memory.append({"question": query, "answer": answer})

    # --- STEP 6: Keep ONLY the last 4 turns ---
    while len(conversation_memory) > MAX_TURNS:
        conversation_memory.pop(0)   # remove the oldest turn

    return answer

print("Memory-aware chat function ready!")


Memory-aware chat function ready!


In [129]:
# First question - introduce a topic
print("Q1: What is BERT?")
print("A1:", chat("What is BERT?"))
print("\n" + "="*50 + "\n")

# Follow-up using "it" - only works if memory works!
print("Q2: What is it used for?")
print("A2:", chat("What is it used for?"))
print("\n" + "="*50 + "\n")

# Another follow-up
print("Q3: How is it different from GPT?")
print("A3:", chat("How is it different from GPT?"))

# Show what's currently in memory
print("\n\nMemory now holds these turns:")
for i, turn in enumerate(conversation_memory):
    print(f"  Turn {i+1}: {turn['question']}")


Q1: What is BERT?
A1: models pre-trained with Ima- geNet


Q2: What is it used for?
A2: to attend to the full input at every output timestep


Q3: How is it different from GPT?
A3: GPT-3 operates in a somewhat different regime.


Memory now holds these turns:
  Turn 1: What is BERT?
  Turn 2: What is it used for?
  Turn 3: How is it different from GPT?


In [130]:
chat("What is GPT-3?")
chat("What is RoBERTa?")
chat("What is T5?")

print("Memory should only hold the LAST 4 turns:")
for i, turn in enumerate(conversation_memory):
    print(f"  Turn {i+1}: {turn['question']}")
print(f"\nTotal turns in memory: {len(conversation_memory)} (should be 4)")


Memory should only hold the LAST 4 turns:
  Turn 1: How is it different from GPT?
  Turn 2: What is GPT-3?
  Turn 3: What is RoBERTa?
  Turn 4: What is T5?

Total turns in memory: 4 (should be 4)


In [131]:
# Each item: the question + a short "ground truth" (correct reference answer you write)
test_data = [
    {
        "question": "What is the main innovation introduced in the Attention Is All You Need paper?",
        "ground_truth": "The Transformer architecture, which relies entirely on self-attention mechanisms and removes recurrence and convolutions."
    },
    {
        "question": "What does the self-attention mechanism do?",
        "ground_truth": "It lets each word attend to all other words in the sequence to capture relationships and context."
    },
    {
        "question": "What does BERT stand for?",
        "ground_truth": "Bidirectional Encoder Representations from Transformers."
    },
    {
        "question": "How is BERT trained?",
        "ground_truth": "Using masked language modeling and next sentence prediction on large unlabeled text."
    },
    {
        "question": "What is GPT-3 known for?",
        "ground_truth": "Being a very large language model (175 billion parameters) capable of few-shot learning without fine-tuning."
    },
    {
        "question": "What is few-shot learning in GPT-3?",
        "ground_truth": "The ability to perform tasks given only a few examples in the prompt, without updating model weights."
    },
    {
        "question": "How does RoBERTa improve on BERT?",
        "ground_truth": "By training longer with more data, larger batches, removing next sentence prediction, and using dynamic masking."
    },
    {
        "question": "What is the main idea of the T5 model?",
        "ground_truth": "Treating every NLP task as a text-to-text problem, where both input and output are text."
    },
    {
        "question": "Why is the Transformer faster to train than RNNs?",
        "ground_truth": "Because self-attention allows parallel processing of the whole sequence, unlike sequential RNNs."
    },
    {
        "question": "Can you summarize what these papers have in common?",
        "ground_truth": "They are all foundational deep learning models for natural language processing based on the Transformer architecture."
    },
]

print(f"Prepared {len(test_data)} test questions.")


Prepared 10 test questions.


In [132]:
# Reset memory so we start the test fresh
conversation_memory = []

questions = []
answers = []
contexts = []        # the retrieved chunks for each question
ground_truths = []

for item in test_data:
    q = item["question"]

    # Retrieve chunks (same as inside chat, but we need them separately for RAGAS)
    query_vector = embed_model.encode([q], convert_to_numpy=True)
    distances, indices = index.search(query_vector, 3)
    retrieved = [all_chunks[idx]["text"] for idx in indices[0]]

    # Get the bot's answer (this also updates memory)
    ans = chat(q)

    # Store everything
    questions.append(q)
    answers.append(ans)
    contexts.append(retrieved)
    ground_truths.append(item["ground_truth"])

    print(f"Q: {q}\nA: {ans}\n{'-'*50}")

print("\nCollected all answers and contexts!")


Q: What is the main innovation introduced in the Attention Is All You Need paper?
A: Attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making...more difficult’.
--------------------------------------------------
Q: What does the self-attention mechanism do?
A: compute a representation of the sequence
--------------------------------------------------
Q: What does BERT stand for?
A: We introduce BERT and its detailed implementation in this section.
--------------------------------------------------
Q: How is BERT trained?
A: BERT pre-training.
--------------------------------------------------
Q: What is GPT-3 known for?
A: improves the quality of text generation and adaptability over smaller models and increases the difficulty of distinguishing synthetic text from human-written text
--------------------------------------------------
Q: What is few-shot learning in GPT-3?
A: GPT-3 takes a step towards test-time sample efficiency closer to that o

In [133]:
# Reset memory so we start the test fresh
conversation_memory = []

questions = []
answers = []
contexts = []        # the retrieved chunks for each question
ground_truths = []

for item in test_data:
    q = item["question"]

    # Retrieve chunks (same as inside chat, but we need them separately for RAGAS)
    query_vector = embed_model.encode([q], convert_to_numpy=True)
    distances, indices = index.search(query_vector, 3)
    retrieved = [all_chunks[idx]["text"] for idx in indices[0]]

    # Get the bot's answer (this also updates memory)
    ans = chat(q)

    # Store everything
    questions.append(q)
    answers.append(ans)
    contexts.append(retrieved)
    ground_truths.append(item["ground_truth"])

    print(f"Q: {q}\nA: {ans}\n{'-'*50}")

print("\nCollected all answers and contexts!")


Q: What is the main innovation introduced in the Attention Is All You Need paper?
A: Attention heads attend to a distant dependency of the verb ‘making’, completing the phrase ‘making...more difficult’.
--------------------------------------------------
Q: What does the self-attention mechanism do?
A: compute a representation of the sequence
--------------------------------------------------
Q: What does BERT stand for?
A: We introduce BERT and its detailed implementation in this section.
--------------------------------------------------
Q: How is BERT trained?
A: BERT pre-training.
--------------------------------------------------
Q: What is GPT-3 known for?
A: improves the quality of text generation and adaptability over smaller models and increases the difficulty of distinguishing synthetic text from human-written text
--------------------------------------------------
Q: What is few-shot learning in GPT-3?
A: GPT-3 takes a step towards test-time sample efficiency closer to that o

In [134]:
!pip install scikit-learn

In [135]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def evaluate():
    results = []

    for i in range(len(questions)):
        # Embed the bot's answer and the ground truth
        ans_vec = embed_model.encode([answers[i]])
        truth_vec = embed_model.encode([ground_truths[i]])

        # 1. ANSWER RELEVANCE: how close is the answer to the correct answer?
        answer_score = float(cosine_similarity(ans_vec, truth_vec)[0][0])

        # 2. CONTEXT RELEVANCE: how close is the answer to the retrieved chunks? (faithfulness)
        context_text = " ".join(contexts[i])
        context_vec = embed_model.encode([context_text])
        context_score = float(cosine_similarity(ans_vec, context_vec)[0][0])

        results.append({
            "question": questions[i],
            "answer_relevance": round(answer_score, 3),
            "faithfulness": round(context_score, 3),
        })

    return results

scores = evaluate()

# Print a neat table
print(f"{'Q#':<4}{'Answer Relevance':<20}{'Faithfulness':<15}")
print("="*40)
for i, r in enumerate(scores):
    print(f"{i+1:<4}{r['answer_relevance']:<20}{r['faithfulness']:<15}")

# Averages
avg_rel = np.mean([r['answer_relevance'] for r in scores])
avg_faith = np.mean([r['faithfulness'] for r in scores])
print("="*40)
print(f"AVERAGE Answer Relevance: {avg_rel:.3f}")
print(f"AVERAGE Faithfulness:     {avg_faith:.3f}")


Q#  Answer Relevance    Faithfulness   
1   0.25                0.629          
2   0.169               0.144          
3   0.236               0.476          
4   0.22                0.59           
5   0.413               0.506          
6   0.252               0.689          
7   0.27                0.526          
8   0.126               0.614          
9   0.071               0.195          
10  0.158               0.132          
AVERAGE Answer Relevance: 0.216
AVERAGE Faithfulness:     0.450
